# Scope sensitivity — does the population boundary change the conclusions? *(population)*

*Read-only informative artifact. This notebook describes whether using a 60-minute
cutoff (`minutes >= 60`) instead of "anyone who stepped on the pitch"
(`minutes > 0`) changes how each signal's rank-correlation with points reads. It
applies no gate, reaches no verdict, and does **not** explain *why* any shift
happens — that is a Diagnostic-tier analysis.*

## Directive questions this notebook answers

- **Determine** whether switching the population from `minutes > 0` to
  `minutes >= 60` changes each signal's rank-correlation with points (a
  per-position shift).
- **Establish** which signals are unaffected by the cutoff and which shift a lot.

(*Explaining* why a signal's association shifts across the two population
definitions — and what scope downstream should therefore prefer — is a Diagnostic-tier
question, deferred; see the closing note.)

This is the closing descriptive question of the `population/` layer: having
described the minutes landscape (`minutes_distribution.ipynb`), the target across
minutes-bands (`points_by_minutes_band.ipynb`), and each signal across minutes-bands
(`signals_by_minutes_band.ipynb`), the last description is whether the **choice between the
two population definitions** moves the measured signal–points association.

## Setup

Load the mart and restrict to the **whole season** (GW 1 to the latest completed
GW). Unlike the sibling notebooks we **do not pre-filter minutes** — the
dual-scope statistic derives *both* populations itself: the **filtered** scope
(`minutes >= 60`, the qualified-start population) and the **minimal** scope
(`minutes > 0`, participation). We keep every row with a recorded `minutes`
value so the comparison is over the same underlying weeks.

We **do** exclude double-gameweeks (`is_dgw == False`), matching the sibling
notebooks' single-fixture base population so the scopes are not distorted by
fixture-doubled (~180-minute) rows.

The robustness machinery lives in `research/foundation/population/robustness.py`
(`compute_dual_scope_rho`, `classify_population_robustness`); geometry per scope
is computed with the shared shape kernels. The notebook only orchestrates.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.foundation.population.robustness import (
    compute_dual_scope_rho,
    classify_population_robustness,
    RHO_SHIFT_STABLE_THRESHOLD,
)
from research.kernels.descriptive.binning import select_bucketing_scheme, bin_analysis
from research.kernels.diagnostic.shape import classify_geometry

try:
    _result = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _result = load_mart()

STUDY_GW_MIN = 1
STUDY_GW_MAX = _result.data_cutoff_gw
TARGET = "total_points"
MINUTES_THRESHOLD = 60   # the 60-minute mark (FPL scoring changes here)

mart = _result.mart
# We do NOT pre-filter minutes (compute_dual_scope_rho derives BOTH scopes itself:
# filtered minutes>=60, minimal minutes>0). We DO exclude DGW for now
# (is_dgw == False) to match the sibling notebooks' single-fixture base population.
df = mart[mart["gw"].between(STUDY_GW_MIN, STUDY_GW_MAX)].copy()
df = df[df["minutes"].notna()].copy()
df = df[df["is_dgw"] == False].copy()

POSITIONS = ["GK", "DEF", "MID", "FWD"]

# Candidate signals: the same performance-signal universe as structure/signal.ipynb
# (numeric per-GW signals; exclude identity / market / structural / context / rolling).
_EXCLUDE = {
    "player_id", "gw", "position_code", "team_id",
    "purchase_price", "minutes", "total_points",
    "is_bgw", "is_dgw", "is_warmup_gw",
    "fdr_avg", "xgc", "home_count", "away_count", "fixture_count",
    "is_live", "is_next", "is_previous", "finished",
    "transfers_in", "transfers_out", "ownership_count",
}
SIGNALS = sorted(
    c for c in df.select_dtypes(include="number").columns
    if c not in _EXCLUDE and "_roll" not in c and not c.endswith("_trend")
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

print(f"Study range: GW {STUDY_GW_MIN} - GW {STUDY_GW_MAX} (whole season, from mart data_cutoff_gw)")
print(f"Boundary under test: minutes >= {MINUTES_THRESHOLD} (filtered) vs minutes > 0 (minimal)")
print(f"Rows (minutes present, DGW excluded): {len(df):,}")
print(f"Signals ({len(SIGNALS)}):", SIGNALS)

Study range: GW 1 - GW 38 (whole season, from mart data_cutoff_gw)
Boundary under test: minutes >= 60 (filtered) vs minutes > 0 (minimal)
Rows (minutes present, DGW excluded): 28,929
Signals (24): ['assists', 'bonus', 'bps', 'clean_sheets', 'clearances_blocks_interceptions', 'creativity', 'defensive_contribution', 'goals_conceded', 'goals_scored', 'ict_index', 'influence', 'own_goals', 'penalties_missed', 'penalties_saved', 'recoveries', 'red_cards', 'saves', 'starts', 'tackles', 'threat', 'xa', 'xg', 'xgi', 'yellow_cards']


## (a) Delta-rho across population scopes

**What we measure** — for every (signal, position) pair, the Spearman rho
between the signal and `total_points` under each scope, and the **rho shift**
`|rho(minutes>=60) - rho(minutes>0)|`, via `compute_dual_scope_rho`. A pair is
*testable* only when both scopes clear the minimum-n floor; otherwise rho is
`None` (reported as a non-testable row).

**What it means** — the rho shift is the direct answer to "does tightening the
population to qualified starts change how this signal relates to scoring?" A
shift near zero means the association is the same whether or not cameos are
included — the boundary is a descriptive choice, not one that moves the result.

**What it doesn't mean** — this is **not** a re-decision of *where* the boundary
sits (that is `points_by_minutes_band.ipynb`) and **not** a causal claim about
minutes. It is season-pooled with DGW excluded. The rho is a measured
association, not a predictive verdict — no gate is applied.

**Guiding question (directive)** — *Determine whether re-scoping `minutes > 0`
→ `minutes >= 60` changes the signal–target association, per signal-position.*

In [2]:
dual = compute_dual_scope_rho(
    df, SIGNALS, POSITIONS, target=TARGET, minutes_threshold=MINUTES_THRESHOLD,
)
dual["position"] = pd.Categorical(dual["position"], categories=POSITIONS, ordered=True)
dual = dual.sort_values(["signal", "position"]).reset_index(drop=True)
display(dual)

testable = dual["rho_shift"].notna()
print(f"\nTestable (signal, position) pairs: {int(testable.sum())} of {len(dual)}")
if testable.any():
    shifts = dual.loc[testable, "rho_shift"].abs()
    print(f"|rho_shift|: max={shifts.max():.4f}, mean={shifts.mean():.4f}, "
          f"median={shifts.median():.4f}")
    print(f"share with |rho_shift| < {RHO_SHIFT_STABLE_THRESHOLD}: "
          f"{(shifts < RHO_SHIFT_STABLE_THRESHOLD).mean() * 100:.1f}%")

,signal,position,n_filtered,n_minimal,rho_filtered,rho_minimal,rho_shift
0,assists,GK,737,747,0.0833,0.0835,0.0002
1,assists,DEF,2944,3845,0.2622,0.2766,0.0144
2,assists,MID,3184,5208,0.4863,0.4556,0.0307
3,assists,FWD,742,1390,0.3692,0.3403,0.0289
4,bonus,GK,737,747,0.5381,0.5352,0.0029
...,...,...,...,...,...,...,...
91,xgi,FWD,742,1390,0.4849,0.6248,0.1399
92,yellow_cards,GK,737,747,-0.1400,-0.1342,0.0058
93,yellow_cards,DEF,2944,3845,-0.2314,-0.1997,0.0317
94,yellow_cards,MID,3184,5208,-0.2558,-0.1317,0.1241



Testable (signal, position) pairs: 84 of 96
|rho_shift|: max=0.6074, mean=0.1252, median=0.0925
share with |rho_shift| < 0.1: 51.2%


## (b) Robustness classification, per (signal, position)

**What we measure** — combine the rho shift with a per-scope **geometry** check
(relationship shape via the shared binning + shape kernels) and classify each
pair via `classify_population_robustness` into `{stable, scope_sensitive,
untested}`: `stable` = small rho shift *and* unchanged geometry;
`scope_sensitive` = a material rho shift *or* a geometry change; `untested` =
insufficient data in either scope.

**What it means** — this is the per-cell answer to "is this signal's
relationship to scoring robust to the population boundary?" Pairs that come back
`stable` are unaffected by the cutoff; `scope_sensitive` pairs are the ones whose
link to points changes once cameos are removed. The mix of the two is the whole
point of the readout.

**What it doesn't mean** — the thresholds (`rho_shift < 0.10` stable;
`<= 0.25` scope_sensitive) are operational heuristics, not significance tests.
A `scope_sensitive` flag driven only by a **geometry label change on a sparse
signal** is classification noise, not association instability — read those
against the rho shift in (a), which is the primary robustness measure.

**Guiding question (directive)** — *Establish which (signal, position) pairs are
scope-robust versus scope-sensitive.*

In [3]:
filtered = df[df["minutes"] >= MINUTES_THRESHOLD]
minimal = df[df["minutes"] > 0]


def _geometry(scope_df, signal, position):
    """Relationship geometry of (signal -> total_points) on one scope/position."""
    series = scope_df.loc[scope_df["position"] == position, signal].dropna()
    scheme = select_bucketing_scheme(series, signal_name=signal)
    bin_stats, _flag = bin_analysis(scope_df, signal, TARGET, position, scheme)
    if bin_stats is None:
        return "unassessable"
    return classify_geometry(bin_stats)


rows = []
for _, r in dual.iterrows():
    sig, pos = r["signal"], r["position"]
    geom_m = _geometry(minimal, sig, pos)
    geom_f = _geometry(filtered, sig, pos)
    robustness = classify_population_robustness(
        r["rho_filtered"], r["rho_minimal"], geom_f, geom_m,
    )
    rows.append({
        "signal": sig, "position": pos,
        "rho_shift": r["rho_shift"],
        "geometry_minimal": geom_m, "geometry_filtered": geom_f,
        "geometry_changed": geom_f != geom_m,
        "population_robustness": robustness,
    })

robustness_df = pd.DataFrame(rows)
robustness_df["position"] = pd.Categorical(robustness_df["position"], categories=POSITIONS, ordered=True)
robustness_df = robustness_df.sort_values(["signal", "position"]).reset_index(drop=True)
display(robustness_df)

print("\nRobustness classification counts:")
print(robustness_df["population_robustness"].value_counts().to_string())

scope_sensitive = robustness_df[robustness_df["population_robustness"] == "scope_sensitive"]
if len(scope_sensitive):
    print("\nscope_sensitive pairs (boundary changes the read here) -- inspect rho_shift vs geometry:")
    display(scope_sensitive[["signal", "position", "rho_shift", "geometry_changed",
                             "geometry_minimal", "geometry_filtered"]])
else:
    print("\nNo scope_sensitive pairs -- every testable pair is robust to the 60-minute boundary.")

,signal,position,rho_shift,geometry_minimal,geometry_filtered,geometry_changed,population_robustness
0,assists,GK,0.0002,indeterminate,indeterminate,False,stable
1,assists,DEF,0.0144,monotonic_positive,monotonic_positive,False,stable
2,assists,MID,0.0307,monotonic_positive,monotonic_positive,False,stable
3,assists,FWD,0.0289,monotonic_positive,monotonic_positive,False,stable
4,bonus,GK,0.0029,monotonic_positive,monotonic_positive,False,stable
...,...,...,...,...,...,...,...
91,xgi,FWD,0.1399,monotonic_positive,monotonic_positive,False,scope_sensitive
92,yellow_cards,GK,0.0058,indeterminate,indeterminate,False,stable
93,yellow_cards,DEF,0.0317,indeterminate,indeterminate,False,stable
94,yellow_cards,MID,0.1241,indeterminate,indeterminate,False,scope_sensitive



Robustness classification counts:
population_robustness
scope_sensitive    44
stable             40
untested           12

scope_sensitive pairs (boundary changes the read here) -- inspect rho_shift vs geometry:


,signal,position,rho_shift,geometry_changed,geometry_minimal,geometry_filtered
7,bonus,FWD,0.1634,False,monotonic_positive,monotonic_positive
10,bps,MID,0.1080,False,monotonic_positive,monotonic_positive
14,clean_sheets,MID,0.1356,False,indeterminate,indeterminate
15,clean_sheets,FWD,0.3019,False,indeterminate,indeterminate
17,clearances_blocks_interceptions,DEF,0.1769,True,monotonic_positive,non_monotonic
18,clearances_blocks_interceptions,MID,0.2325,False,unassessable,unassessable
19,clearances_blocks_interceptions,FWD,0.2881,False,monotonic_positive,monotonic_positive
21,creativity,DEF,0.1766,True,monotonic_positive,non_monotonic
22,creativity,MID,0.2907,False,monotonic_positive,monotonic_positive
23,creativity,FWD,0.2434,False,monotonic_positive,monotonic_positive


## What the scope-sensitivity readout says

Plain-language description (not a verdict, not a gate, not an explanation):

**The short version.** Switching the population from "anyone who played"
(`minutes > 0`) to "played 60+ minutes" (`minutes >= 60`) changes the
signal–points rank-correlation for a lot of the raw single-game stats — **44 of
84** testable signal-position pairs shift, by up to **0.61** (on a 0-1 scale). For
these stats the population choice is **not** a harmless technicality. (12 more
pairs are `untested` — too few rows in one of the two groups to compare, mostly
sparse stats at thin positions like GK/FWD.)

**Which signals move, and which don't.** The big movers are `starts` (avg shift
0.41), the defensive-action counts (`defensive_contribution`, `recoveries`,
`clearances_blocks_interceptions`, `tackles`), `goals_conceded`, and the per-match
ICT totals (`creativity`, `xa`, `xgi`). The stats that barely move are the rare or
small-magnitude ones — a `goal`, an `assist`, `bonus`, `saves`, `penalties_saved`
— all under 0.08. (`signals_by_minutes_band.ipynb` separately describes how these same
signals sit across minutes-bands.)

**What this does NOT say.** It does not explain *why* these associations shift, and
it does not conclude that either population is "correct". A shift could reflect how
the signal co-moves with minutes, differences in who is in each population, or
other structure — separating those is a **Diagnostic-tier analysis**, not done here.

**Note on an earlier reading.** A prior robustness check reported little change
across the cutoff. That check and this one looked at **different signal sets** —
that one the smoothed rolling-average form signals, this one the raw single-game
mart stats — so the two describe different inputs, not contradicting each other.

**Deferred follow-up (Diagnostic-tier analysis, not built here).** Explaining the shift — e.g.
describing each signal's association *adjusted for* minutes, normalising counts to
per-90, or reading the association *within* minutes-bands — is a diagnostic /
inferential step for a later phase. The tooling already exists
(`research/kernels/inferential/resampling.py`,
`research/kernels/diagnostic/conditioning.py`).

All figures are whole-season, season-pooled, with **DGW excluded**
(`is_dgw == False`). The correlations here are descriptive associations in this
season's data, not predictive verdicts — no gate is applied.